# Lesson 3.1 - Supervised Learning

## Objectives
- Formalize regression and classification tasks.
- Compare linear, tree, bagging, boosting, and XGBoost models.
- Discuss model choice tradeoffs for business constraints.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBClassifier, XGBRegressor


## Section A - Classification (Breast Cancer)


In [ ]:
cancer = load_breast_cancer(as_frame=True)
Xc, yc = cancer.data, cancer.target
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.25, random_state=42, stratify=yc)

models_clf = {
    "log_reg": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=2000))]),
    "knn": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=7))]),
    "decision_tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=250, random_state=42),
    "grad_boost": GradientBoostingClassifier(random_state=42),
    "xgboost": XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, eval_metric="logloss", random_state=42),
}

rows = []
for name, m in models_clf.items():
    m.fit(Xc_train, yc_train)
    p = m.predict(Xc_test)
    proba = m.predict_proba(Xc_test)[:, 1]
    rows.append({"model": name, "accuracy": accuracy_score(yc_test, p), "f1": f1_score(yc_test, p), "roc_auc": roc_auc_score(yc_test, proba)})

clf_results = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
display(clf_results)


## Section B - Regression (Diabetes)


In [ ]:
d = load_diabetes(as_frame=True)
Xr, yr = d.data, d.target
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.25, random_state=42)

models_reg = {
    "linear_reg": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "random_forest": RandomForestRegressor(n_estimators=250, random_state=42),
    "grad_boost": GradientBoostingRegressor(random_state=42),
    "xgboost": XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, random_state=42),
}

rows = []
for name, m in models_reg.items():
    m.fit(Xr_train, yr_train)
    p = m.predict(Xr_test)
    rows.append({"model": name, "rmse": mean_squared_error(yr_test, p) ** 0.5, "r2": r2_score(yr_test, p)})

reg_results = pd.DataFrame(rows).sort_values("rmse")
display(reg_results)


## Section C - Ensemble Intuition
- Bagging (Random Forest): reduce variance with many trees on bootstraps.
- Boosting (Gradient Boosting/XGBoost): reduce bias by sequential error correction.


## Business Case Studies & Exceptions
- Credit scoring: logistic baseline easier governance, boosting often improves ranking.
- Pricing regression: linear model underfits non-linear effects; boosted trees improve RMSE.
- Exception: powerful ensembles can overfit noisy/high-cardinality features if tuning ignored.


## Interview Questions & Answers
1. **Q:** Supervised learning definition?  
   **A:** Learn mapping from labeled inputs to outputs.
2. **Q:** Regression vs classification?  
   **A:** Continuous vs categorical target.
3. **Q:** What is bagging?  
   **A:** Bootstrap aggregation of models.
4. **Q:** What is boosting?  
   **A:** Sequential weak learners correcting previous errors.
5. **Q:** Random Forest vs Gradient Boosting?  
   **A:** RF parallel variance reduction; GB sequential bias reduction.
6. **Q:** Why XGBoost popular?  
   **A:** Strong tabular performance + regularization + optimized system design.
7. **Q:** When choose linear model?  
   **A:** Interpretability and low-latency constraints.
8. **Q:** Why baseline needed?  
   **A:** Anchor to measure value of complexity.
9. **Q:** Main risk with boosting?  
   **A:** Overfitting without tuning.
10. **Q:** Why multi-metric comparison?  
    **A:** Single metric hides tradeoffs.
